#### 로컬 임베딩 모델 사용하기

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from glob import glob

for g in glob('./data/*.pdf'):
    print(g)


./data\2040_seoul_plan.pdf
./data\OneNYC_2050_Strategic_Plan.pdf


In [ ]:
# 임베딩 모델 설치
# pip install langchain-huggingface

# 파이토치 설치하기(option)
# CUDA를 지원하는 NVIDIA 그래픽 카드를 갖고 있다면 파이토치를 설치
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [7]:
# 청크 분할하기
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """주어진 PDF 파일을 읽고 텍스트를 분할합니다.
    매개변수:
        pdf_path (str): 파일의 경로
        chunk_size (int, 선택적): 각 텍스트 청크의 크기. 기본값은 1000입니다.
        chunk_overlap(int, 선택적): 청크 간의 중첩 크기. 기본값은 100입니다.
    반환값:
        list: 분할된 텍스트 청크의 리스트    
    """
    print(f'PDF: {pdf_path} ------------------')

    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    splits = text_splitter.split_documents(data_from_pdf)
    print(f'Number of splits: {len(splits)}')

    return splits

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs = {'device': 'cpu'},               # cuda 지원하면 cuda, 아닌 경우 cpu
    encode_kwargs = {'normalize_embeddings': True}, # 임베딩 정규화
)

c:\workspaces\ai_agent\src\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 35544.95it/s]


In [ ]:
# 임베딩 모델 테스트
embeddings.embed_documents('안녕하세요.')

In [11]:
# pdf 파일 읽고 텍스트 임베딩하기
from langchain_chroma import Chroma
import os

persist_directory = './chroma_store'  # 벡터 DB

if os.path.exists(persist_directory):
    print('Loading existing Chroma store')

    vectorstore = Chroma(  # 읽어 올때
        persist_directory=persist_directory,
        embedding_function=embeddings
    )

else:
    print('Creating new Chroma store')

all_splits= []
for g in glob('./data/*.pdf'):
    all_splits.extend(read_pdf_and_split_text(g))

    print(f'Total number of splits: {len(all_splits)}')

    vectorstore = Chroma.from_documents(  # 저장(생성)할 때 : 최초 한번만 호출되어야 함
        documents=all_splits,
        embedding=embeddings,  # BAAI/bge-m3 모델
        persist_directory=persist_directory
    )

Creating new Chroma store
PDF: ./data\2040_seoul_plan.pdf ------------------
Number of splits: 308
Total number of splits: 308
PDF: ./data\OneNYC_2050_Strategic_Plan.pdf ------------------
Number of splits: 1023
Total number of splits: 1331


In [12]:
# 리트리버 테스트(검색기)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5}) # 5개의 chunk 데이터를 반환

chunks = retriever.invoke('서울시 쓰레기 저감 정책')

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)


{'producer': 'Hancom PDF 1.3.0.542', 'creationdate': '2024-12-12T18:16:11+09:00', 'creator': 'Hwp 2020 11.0.0.5178', 'source': './data\\2040_seoul_plan.pdf', 'pdfversion': '1.4', 'total_pages': 205, 'moddate': '2024-12-12T18:16:11+09:00', 'page': 64, 'author': 'SI', 'page_label': '65'}
제3절 2040 서울도시기본계획 7대 목표57Ÿ특히, 서울시는 현재 온실가스 배출량의 90%를 차지하고 있는 건물과 수송 부문 감축을 위해 적극적인 대책을 마련하고 있다.-2026년까지 건물 에너지효율화사업 100만 호를 추진, 건물온실가스총량제, 신규건물 제로에너지 건물(ZEB) 의무화를 통해 기존 및 신규건물의 제로에너지화를 촉진-수송 부문 배출감축을 위해 전기차 비중을 2026년까지 10%(’21년 5.2만 대 → ’26년 40만 대)로 확대하고, 22만기의 충전 인프라 구축을 계획 중(’21년 2만기) 시민 생활과 안전을 위한 기후위기 대응 요구 심화Ÿ서울은 인구, 시설 등이 밀집해 있는 대도시로 기온 상승, 폭염, 집중호우, 태풍, 한파 등 극한 기후 현상이 더욱 빈번하게 발생할 것으로 전망된다. 이러한 기후위험은 서울시민의 일상생활과 안전을 크게 위협할 수 있다. 따라서 시민의 일상을 보호하고 도시의 회복력을 강화하기 위한 적극적인 기후위기 대응 전략이 필요하다.Ÿ특히 다양화·복합화되는 재난안전사고에 대응해 전통적인 자연·사회재난의 범주뿐 아니라, 신종 복합재난까지 대비할 수 있는 다면적인 대응체계 마련이 요구된다.2) 추진전략탄소중립·기후위기 적응대책은 시의 모든 정책과 사업의 주요 원칙으로 고려Ÿ탄소중립은 돌이키기 어려운 기후재난을 막기 위한 국제사회와의 약속이기에 시의 모든 정책과 사업의 결정 과정에서 주요하게 고려해야 할 포괄적 ‘원칙’으

In [ ]:
# 구현한 RAG를 탑재시킨 에이전트 생성
# RAG을 tool로 등록
